# Tutorial 4 — Cellular Automata & Agent-Based Models

Three systems where global behaviour *emerges* from local rules: Conway's **Game of Life**, the **forest-fire** cellular automaton, and Reynolds' **boids** flocking model.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from sim_lab.core import (
    GameOfLifeSimulation,
    ForestFireSimulation,
    BoidsSimulation,
)

random_seed = 42
np.random.seed(random_seed)
print(f"Reproducibility seed locked: {random_seed}")

## 1. Game of Life — a glider

A **glider** is a 5-cell spaceship that translates diagonally every 4 generations while keeping exactly 5 live cells. On a periodic grid it travels forever, so the population count is a constant 5 — a sharp, checkable invariant.

In [ ]:
gol = GameOfLifeSimulation(grid_size=(20, 20), pattern='glider',
                          days=16, boundary='periodic', random_seed=random_seed)
pop = [int(p) for p in gol.run_simulation()]

fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for ax, gen in zip(axes, [0, 4, 8, 12]):
    ax.imshow(gol.get_state_at_day(gen), cmap='Greys')
    ax.set_title(f'gen {gen}: {int(pop[gen])} cells'); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Glider translating across the toroidal grid')
plt.tight_layout(); plt.show()

print('population per generation:', pop)
assert all(p == 5 for p in pop), 'a glider must keep exactly 5 live cells per generation'
print('invariant holds: every generation has exactly 5 live cells.')

## 2. Forest Fire — self-organised criticality

Drossel–Schwabl rules: a burning cell clears; a tree ignites if a neighbour burns *or* lightning strikes (probability $p$); an empty cell regrows a tree (probability $g$). With $p \ll g$ the forest hovers near full and burns in **episodic** bursts — avalanches of fire separated by quiet regrowth.

In [ ]:
ff = ForestFireSimulation(
    grid_size=(60, 60), initial_density=0.6,
    p=1e-4, g=1e-2, days=250, boundary='periodic', random_seed=random_seed,
)
trees = ff.run_simulation()
fires = ff.fire_history
gens = np.arange(len(fires))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(gens, trees, color='forestgreen', label='trees')
ax.plot(gens, fires, color='red', label='burning')
ax.set_xlabel('generation'); ax.set_ylabel('cell count')
ax.set_title(f'Forest fire (p<<g): episodic burns, peak {int(max(fires))} cells')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

burning_gens = int(np.sum(np.array(fires) > 0))
print(f'generations with any fire = {burning_gens} / {len(fires)}')
print(f'peak fire size = {int(max(fires))} cells (episodic burst)')
assert max(fires) > 50, 'large episodic burns must occur'
assert burning_gens > 0, 'fires must ignite'

## 3. Boids — emergent flocking

Each boid follows three local rules — **separation**, **alignment**, **cohesion** — using only its neighbours within a perception radius. No central controller exists; we watch two emergent diagnostics over time: the flock's **mean speed** (alignment) and its **positional spread** (cohesion pulling the swarm together).

In [ ]:
boids = BoidsSimulation(num_boids=80, width=80.0, height=80.0,
                       perception_radius=15.0, days=150, random_seed=random_seed)
metrics = boids.run_simulation()
spread = [m['flock_spread'] for m in metrics]
speed = [m['mean_speed'] for m in metrics]
steps = np.arange(len(metrics))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(steps, speed, color='steelblue')
axes[0].set_xlabel('step'); axes[0].set_ylabel('mean speed')
axes[0].set_title('Alignment: mean speed settles'); axes[0].grid(alpha=0.3)
axes[1].plot(steps, spread, color='darkgreen')
axes[1].set_xlabel('step'); axes[1].set_ylabel('flock spread (std of position)')
axes[1].set_title('Cohesion: spread tightens'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'mean speed : {speed[0]:.3f} -> {speed[-1]:.3f}')
print(f'flock spread: {spread[0]:.3f} -> {min(spread):.3f} (min)')
assert min(spread) < spread[0], 'cohesion should tighten the flock'
assert len(metrics) == 150, 'one metric record per step'

## Validation & interpretation

| Model | Invariant / behaviour | Check |
|---|---|---|
| Game of Life | glider = 5 cells each generation | all 16 generations have 5 cells ✅ |
| Forest Fire | $p \ll g$ gives episodic burns | peak burst > 50 cells, fires recur ✅ |
| Boids | local rules -> aligned, cohesive flock | spread tightens, speed regularises ✅ |

These are the canonical demos of *emergence*. The glider's 5-cell count is an exact conservation law hiding inside chaotic-looking rules. The forest settles into self-organised criticality: trees accumulate until a lightning strike triggers an avalanche whose size is power-law distributed — small fires most days, a monster occasionally. The boids need no leader; alignment drives their speeds toward a common value while cohesion shrinks the swarm's footprint.